# M5-v2: ID 협업점수로 hard negative 선택 (Dunnhumby)

M2가 올린 경제적 후보를 M4가 직접 hard negative로 선택해 억누르는 경로를 분리합니다. Hard negative는 공동학습된 ID 64차원 점수로 선택하고, BPR 손실은 CLV 3차원을 포함한 전체 점수로 계산합니다. 이전 M5-v1의 5개 대조군은 저장된 결과를 재사용하고, 실제 CLV와 degree-matched CLV 순열 두 arm만 새로 학습합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess

REVIEWED_SHA = '4737718851babe7ef9d03d9593f882207d0548b6'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')  # 이전 checkout 내부를 현재 폴더로 둔 채 삭제하지 않는다.
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    clone_result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if clone_result.returncode == 0:
        break
    clone_errors.append(clone_result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', clone_result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_embedding_hard_negative import (
    configure_m5_run,
    preflight_summary,
    run_m5_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_m5_run(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_id_selected_hard_negative_historical_screen_v2',
    baseline_result_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1',
    previous_m5_result_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_m2_m4_joint_historical_screen_v1',
)
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_m5_screen(cfg)

In [ ]:
from IPython.display import display
import pandas as pd

print('1) 절대지표: 이전 M1·M2·M4·M5, ID 점수 선택 M5-v2, M5-v2 CLV 순열')
display(result_df)
print('2) 대조군별 비교')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) M2×M4 상호작용')
display(pd.DataFrame(result_df.attrs['interaction']))
print('4) 사전 판정')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('5) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))